In [1]:
import requests, csv
from bs4 import BeautifulSoup
import re
import tqdm
import pandas as pd
import numpy as np
from collections import OrderedDict
import tqdm
from urllib.request import Request, urlopen
import os
import json
import undetected_chromedriver as uc
import time
import random
from concurrent.futures import ThreadPoolExecutor, as_completed

In [2]:
from selenium import webdriver
from selenium.webdriver.chrome.options import Options

In [25]:
def parse_json_ld(soup):
    script_tag = soup.find('script', {'type': 'application/ld+json'})
    try:
        return json.loads(script_tag.string) if script_tag else {}
    except:
        return {}

def parse_duration(duration_str):
    if not duration_str: return 0
    duration = str(duration_str).replace("PT", "")
    hours = int(duration.split("H")[0]) if "H" in duration else 0
    minutes_part = duration.split("H")[1] if "H" in duration else duration
    minutes = int(minutes_part.split("M")[0]) if "M" in duration else 0
    return hours * 60 + minutes


def get_production_info(soup):
    companies = soup.find_all('a', {'href': re.compile('company')})
    
    if "Production compan" not in companies[0].text:
        prod_names = []
    else :
        prod_names = [c.text for c in companies if "IMDbPro" not in c.text and c.text != "" and "Production compan" not in c.text]
        
    prod_codes = [str(c).split("company/")[1].split("/?ref")[0] for c in companies if "company/" in str(c)]
        
    return prod_names, prod_codes

def get_cast_and_director(soup):
    sections = soup.select('[class="ipc-page-section ipc-page-section--base ipc-page-section--bp-none"]')

    dir_section = None
    cast_section = None

    for section in sections:
        head = section.find('div', {"class": re.compile("ipc-title__wrapper")})
        
        if dir_section != None and cast_section != None:
            break
        elif "Director" in [a.text.strip() for a in head][0]:
            dir_section = section
            directors = [d.text.strip() for d in dir_section.find_all('a', {"class": re.compile("title-text-big")})]
            dir_codes = [a['href'].split('/')[2] for a in dir_section.find_all('a', {"class": re.compile("title-text-big")})]
        elif [a.text.strip() for a in head][0] == "Cast":
            cast_section = section
            actors = cast_section.find_all('a', {"class": re.compile("title-text-big")})
            cast_names = [a.text.strip() for a in actors]
            cast_codes = [(str(actors[i]).split('/name/')[1]).split('/?ref')[0] for i in range(len(actors))]

            spans = cast_section.findAll('span')
            good_span = [span for span in spans if span.get_text(strip=True) != "/"]

            cast_voice = []
            for idx, span in enumerate(good_span):
                if "ipc-metadata-list-summary-item__t" not in span.get("class", []):
                    continue

                if idx == len(good_span) - 1:
                    cast_voice.append("A")
                    continue

                next_text = good_span[idx + 1].get_text(strip=True).lower()

                if "(voice)" in next_text:
                    cast_voice.append("V")
                elif "uncredited" in next_text:
                    cast_voice.append("U")
                else:
                    cast_voice.append("A")

            mask = [el != "U" for el in cast_voice]
            cast_names = [cast_names[i] for i in range(len(mask)) if mask[i]]
            cast_codes = [cast_codes[i] for i in range(len(mask)) if mask[i]]
            cast_voice = [cast_voice[i] for i in range(len(mask)) if mask[i]]

            if len(cast_names) > 30:
                cast_names, cast_codes, cast_voice = cast_names[:30], cast_codes[:30], cast_voice[:30]

    return cast_names, cast_codes, cast_voice, directors, dir_codes


# --- FONCTION PRINCIPALE ---

def movie_imdb_full_logic(imdb_id, french_title):
    
    # 1. PAGE PRINCIPALE (Infos, Budget, Directors)
    url_main = f"https://www.imdb.com/title/{imdb_id}/?language=en-US"
    driver.get(url_main)
    time.sleep(random.uniform(4, 6))
        
    soup_main = BeautifulSoup(driver.page_source, "html.parser")
    info_main = parse_json_ld(soup_main)
        
    # 2. PAGE FULL CREDITS (Pour avoir le Top 30)
    url_credits = f"https://www.imdb.com/title/{imdb_id}/fullcredits?language=en-US"
    driver.get(url_credits)
    time.sleep(random.uniform(3, 5))
        
    soup_credits = BeautifulSoup(driver.page_source, "html.parser")

    # --- EXTRACTION ---
    title = info_main.get("name", "").replace("&amp;", "&").replace("&apos;", "'")

    # Cast (depuis la fonction dédiée sur soup_credits)
    cast_names, cast_codes, cast_roles, directors, dir_codes = get_cast_and_director(soup_credits)

    # Le reste de la logique (Budget, Pays, Langues, Prods)
    # Budget
    try:
        b_section = soup_main.find_all('span', {'class': 'ipc-metadata-list-item__list-content-item'})
        budget_text = [s.text for s in b_section if "estimated" in s.text][0]
        parts = budget_text.replace("\u202f", "").replace("\xa0", " ").split(" ")
        budget, bud_currency = int(parts[0]), parts[1]
    except Exception:
        budget, bud_currency = 0, ""

    # Productions
    prod_names, prod_codes = get_production_info(soup_main)

    # Rating, Year, Genres, Pays, Langues
    rating = info_main.get("aggregateRating", {}).get("ratingValue", "")
    num_rate = info_main.get("aggregateRating", {}).get("ratingCount", "")

    year_m = re.search(r'\((\d{4})\)', soup_main.title.text)
    year = int(year_m.group(1)) if year_m else None

    duration = parse_duration(info_main.get("duration", ""))
    genres = info_main.get("genre", [])

    country = [c.text for c in soup_main.find_all('a', {'href': re.compile('country_of_origin')})] or ["No Info"]
    country = ["USA" if c == "United States" else "UK" if c == "United Kingdom" else c for c in country]
    language = [l.text for l in soup_main.find_all('a', {'href': re.compile('primary_language')})] or ["No Info"]

    return (
            imdb_id, french_title, title, year, directors, dir_codes, 
            cast_names, cast_codes, cast_roles, genres, duration, 
            country, language, prod_names, prod_codes, 
            rating, num_rate, budget, bud_currency
        )


In [13]:
id_film = "tt35591633"
resultat = movie_imdb_full_logic(id_film, "test")
print(resultat)

[tt35591633] Page principale...
[tt35591633] Page crédits (Cast Top 30)...
dir-cast ok
budget ok
prod ok
rate ok
('tt35591633', 'test', 'Alter ego', 2026, ['Nicolas Charlet', 'Bruno Lavaine', 'Nicolas & Bruno'], ['nm3299329', 'nm3298144', 'nm1335124'], ['Laurent Lafitte', 'Blanche Gardin', 'Olga Kurylenko', 'Marc Fraize', 'Zabou Breitman', 'Bertrand Goncalves', 'Ahmed Hammadi Chassin', 'Giovanni Pucci', 'Léo Garcia', 'Olivier Brabant', 'Olivier Bayart', 'Emmanuel Plovier', 'Hervé Hague', 'Frédéric Amière', 'Jean-Paul Mongin', 'Sabine Lamotte Debuisson', 'Thierry Debuisson', 'Vincent Corigliano', 'Tony Boussadia', 'Arnaud Conter', 'Florent Dubois', 'Lionel Fibleuil', 'Béatrice Gaillard', 'Omar Planas Calas', 'Morgan Grumelard', 'Julie Macioszek Ringart', 'Mame Diarra Sow Conter', 'Charlotte Vasseur', 'Florence Beck', 'Patricia Mainget Ringart'], ['nm0480850', 'nm3226240', 'nm1385871', 'nm8988411', 'nm0951456', 'nm8202425', 'nm11845313', 'nm10096895', 'nm16996314', 'nm0102617', 'nm247064

In [4]:
from concurrent.futures import ThreadPoolExecutor, as_completed

def get_movies_multithread(imdb_ids, max_workers=10):
    results = []

    with ThreadPoolExecutor(max_workers=max_workers) as executor:
        futures = {executor.submit(movie_imdb, imdb_id): imdb_id for imdb_id in imdb_ids}

        for future in as_completed(futures):
            imdb_id = futures[future]

            data = future.result()
            results.append(data)

    return results


In [12]:
# import movies to scrap
df = pd.read_csv('scrap_it.csv', encoding="ISO-8859-1", sep=";")

In [13]:
df["query"] = df.apply(lambda x : (x["imdb"],x["title"]), axis=1)

In [14]:
db_df = pd.read_csv('db_backup/movies_db.csv', encoding="ISO-8859-1", sep=";")

In [15]:
imdb_db = db_df["movie_id"].values
final_imdb = [(i,j) for (i,j) in list(df["query"].values) if i not in imdb_db]

In [52]:
movies = []

In [53]:
options = uc.ChromeOptions()
    # options.add_argument('--headless') 
    
driver = uc.Chrome(options=options, version_main=145)
driver.execute_cdp_cmd("Network.enable", {})

driver.execute_cdp_cmd("Network.setExtraHTTPHeaders", {
        "headers": {
            "Accept-Language": "en-US,en;q=0.9"
        }
    })

{}

In [54]:
try :
    for (i, j) in tqdm.tqdm(final_imdb):
        movies.append(movie_imdb_full_logic(i, j))
finally :
    driver.quit()

100%|██████████████████████████████████████████| 34/34 [11:03<00:00, 19.53s/it]


In [55]:
movies

[('tt35404853',
  'Ceux qui comptent',
  'Ceux qui comptent',
  2026,
  ['Jean-Baptiste Léonetti'],
  ['nm1515266'],
  ['Sandrine Kiberlain',
   'Pierre Lottin',
   'Louise Labèque',
   'Alexis Rosenstiehl',
   'Alma Ngoc',
   'Melissa Izquierdo',
   'Mogamed Bechiev',
   'Frans Boyer',
   'Matthieu Jordan',
   'Olivier-Pierre Richard',
   'Florent Bigot de Nesles',
   'Louise Charnalet',
   'Bruno Paviot',
   'Quitterie Picamoles',
   'Kevin Marvinio',
   'Robert Moundi',
   'Philéas Bourdon',
   'Lucas El Bali',
   'Dune Miki',
   'Lison Thébault',
   'Laura Mounier',
   'Ianis Bonnel',
   'Mila Besson',
   'Marie Laparra'],
  ['nm0452161',
   'nm4544506',
   'nm9737150',
   'nm16533347',
   'nm18329093',
   'nm15772434',
   'nm12537581',
   'nm1533577',
   'nm17690655',
   'nm12153504',
   'nm1039800',
   'nm10588346',
   'nm0667664',
   'nm5191162',
   'nm16590404',
   'nm11325878',
   'nm18352280',
   'nm11748918',
   'nm18352281',
   'nm18352279',
   'nm18352282',
   'nm15896188'

In [ ]:
movies = get_movies_multithread(final_imdb, max_workers=10)

In [56]:
movie_df = pd.DataFrame(movies,columns = ["movie_id","french_title","original_title","year","director","director_id",\
                                          "actor","actor_id","status","genre","duration","country",\
                                          "language","production","prod_id","rating","num_rate",\
                                          "budget","currency"])

In [57]:
# keep movies with a rating (the other are probably not available)
movie_keep = movie_df.drop(movie_df[movie_df["rating"] == ""].index).reset_index(drop=True)
too_soon = pd.DataFrame(movie_df[["movie_id","french_title"]].drop(movie_df[movie_df["rating"] != ""].index).values, \
            columns= ['imdb_id', 'title']).set_index("imdb_id")

In [58]:
# save scraped movies with no ratings
if len(too_soon) > 0:
    f_scrap = pd.read_csv('too_soon.csv', encoding="ISO-8859-1", sep=";").set_index("imdb_id")
    too_soon = pd.concat([f_scrap, too_soon])
    too_soon.to_csv('too_soon.csv', encoding="ISO-8859-1", sep=";")

In [59]:
changer = {"$US":1, "$CA":0.72, "$AU":0.65, "€":1.08, "£GB":1.29, "₩":0.00073, "₹":0.012, "CNY":0.14, "RUR":0.011, \
          "CZK":0.043, "NOK":0.092, "BDT":0.0085, "HKD":0.13, "R$":0.162, "CHF":0.8, "MYR":0.24}

In [60]:
movie_keep["def_budget"] = movie_keep["budget"] * movie_keep["currency"].apply(lambda x : changer[x] if x != "" else 1)

In [61]:
movie_keep["def_budget"] = movie_keep["def_budget"].apply(lambda x: int(x) if x !=0 else "")

In [62]:
modifs = ["actor","country","director","genre","language","production"]

In [63]:
for mod in modifs:
    movie_keep[f"{mod}s"] = movie_keep[mod].apply(lambda x : " | ".join(x))

In [64]:
movie_keep["saw"] = False
movie_keep["wishlist"] = False

In [65]:
select = movie_keep[["movie_id","saw","wishlist","french_title","original_title","year","directors","actors","genres",\
                     "duration","countrys","languages","productions","rating","num_rate","def_budget"]]

In [66]:
select.to_excel("movies_scraped.xlsx", index=False)

In [68]:
# update movies_db
up_movies = select[["movie_id","french_title","original_title","year","duration","rating","num_rate","def_budget"]]\
                    .set_index("movie_id").rename(columns = {"def_budget":"budget"})
up_movies.to_csv("update_db/up_movies.csv", encoding="ISO-8859-1", sep=";")

In [69]:
# update user_db
up_user = select[["movie_id","saw","wishlist"]].set_index("movie_id")
up_user.to_csv("update_db/up_user.csv", encoding="ISO-8859-1", sep=";")

In [70]:
genre_df = pd.read_csv("db_backup/genres_db.csv", sep=";")

In [71]:
genre_dict = dict(zip(genre_df["name"], genre_df["genre_id"]))
genre_dict[""] = ""

In [72]:
tot_genre_df = movie_keep[["movie_id","genre"]].explode(["genre"], ignore_index=True)
tot_genre_df = tot_genre_df.fillna("")
tot_genre_df["genre_id"] = tot_genre_df["genre"].apply(lambda x: genre_dict[x])
tot_genre_df = tot_genre_df.drop(["genre"], axis=1).drop_duplicates()
tot_genre_df = tot_genre_df[tot_genre_df["genre_id"].notna() & (tot_genre_df["genre_id"].astype(str).str.strip() != "")]

In [73]:
tot_genre_df.set_index("movie_id").to_csv("update_db/up_movie_genre.csv", sep=";")

### Update actors

In [74]:
tot_actor_df = movie_keep[["movie_id","actor", "actor_id", "status"]].explode(["actor", "actor_id", "status"], ignore_index=True)
tot_actor_df["status"] = tot_actor_df["status"].str.replace("A", "", regex=False)
tot_actor_final = tot_actor_df.drop(["actor"], axis=1).drop_duplicates()
tot_actor_final = tot_actor_final[tot_actor_final["actor_id"].notna() & (tot_actor_final["actor_id"].str.strip() != "")]
tot_actor_final.set_index("movie_id").to_csv("update_db/up_movie_actor.csv", sep=";")

In [75]:
act_db = pd.read_csv(f'db_backup/actors_db.csv', encoding="ISO-8859-1", sep=";")

actors_df = tot_actor_df.drop(["movie_id", "status"], axis=1).drop_duplicates()
new_actors = actors_df[~actors_df["actor_id"].isin(act_db["actor_id"])]
new_actors= new_actors[new_actors["actor_id"].notna() & (new_actors["actor_id"].str.strip() != "")]
if len(new_actors) != 0:
    new_actors.columns = ["name","actor_id"]
    new_actors.set_index("actor_id").to_csv("update_db/up_actor.csv", encoding="ISO-8859-1", sep=";")

### Update director and production

In [76]:
db_dict = {"dir":["director","director_id"],"prod":["production","prod_id"]}

In [78]:
for k, v in db_dict.items():
    tot_df = movie_keep[["movie_id",v[0], v[1]]].explode([v[0], v[1]], ignore_index=True)
    tot_df_final = tot_df.drop([v[0]], axis=1).drop_duplicates()
    tot_df_final = tot_df_final[tot_df_final[v[1]].notna() & (tot_df_final[v[1]].str.strip() != "")]
    tot_df_final.set_index("movie_id").to_csv(f"update_db/up_movie_{k}.csv", sep=";")

    val_db = pd.read_csv(f'db_backup/{v[0]}_db.csv', encoding="ISO-8859-1", sep=";")
    
    val_df = tot_df.drop(["movie_id"], axis=1).drop_duplicates()
    new_val = val_df[~val_df[v[1]].isin(val_db[v[1]])]
    if len(new_val) != 0:
        new_val.columns = ["name",v[1]]
        new_val.set_index(v[1]).to_csv(f"update_db/up_{v[0]}.csv", encoding="ISO-8859-1", sep=";")

### Update Country and Language

In [79]:
db_list = ["country", "language"]

In [80]:
def get_or_create_id(name):
    global max_id
    if name not in info_dict.keys():
        max_id += 1
        info_dict[name] = max_id
    return info_dict[name]

In [81]:
for val in db_list:
    val_db2 = pd.read_csv(f'db_backup/{val}_db.csv', encoding="ISO-8859-1", sep=";")
    info_dict = dict(zip(val_db2["name"], val_db2[f"{val}_id"]))
    max_id = max(val_db2[f"{val}_id"])
    
    tot_df2 = movie_keep[["movie_id",val]].explode([val], ignore_index=True)
    tot_df2[f"{val}_id"] = tot_df2[val].apply(get_or_create_id)
    tot_df2.columns = ["movie_id","name",f"{val}_id"]
    tot_df2_final = tot_df2.drop("name", axis=1).drop_duplicates()
    tot_df2_final = tot_df2_final[tot_df2_final[f"{val}_id"].notna() & (tot_df2_final[f"{val}_id"] != "")]
    tot_df2_final.set_index("movie_id").to_csv(f"update_db/up_movie_{val}.csv", sep=";")
    
    val_df2 = tot_df2.drop(["movie_id"], axis=1).drop_duplicates()
    new_val2 = tot_df2[~tot_df2[f"{val}_id"].isin(val_db2[f"{val}_id"])]
    if len(new_val2) != 0:
        new_val2.set_index(f"{val}_id").to_csv(f"update_db/up_{val}.csv", encoding="ISO-8859-1", sep=";")

In [82]:
import sqlalchemy as db
import psycopg2
import pandas as pd
import os

In [83]:
from dotenv import load_dotenv

In [84]:
load_dotenv()

True

In [85]:
DATABASE_URL = os.getenv("DATABASE_URL")

In [86]:
engine = db.create_engine(DATABASE_URL)
connection = engine.connect()

In [87]:
dict_file = {"movies_db":["up_movies", "movie_id", "movies"], "actors_db":["up_actor", "actor_id", "actors"], \
             "country_db":["up_country", "country_id", "country"], "director_db":["up_director", "director_id","directors"], \
             "genres_db":["up_genre", "genre_id", "genres"], "language_db":["up_language", "language_id", "language"], \
             "production_db":["up_production", "prod_id", "production"],"movie_actor_db_2":["up_movie_actor", "movie_id", "movie_actor"], \
             "movie_country_db":["up_movie_country", "movie_id", "movie_country"], "movie_dir_db":["up_movie_dir", "movie_id", "movie_director"], \
             "movie_genre_db":["up_movie_genre", "movie_id", "movie_genre"], "movie_language_db":["up_movie_language", "movie_id", "movie_language"], \
             "movie_prod_db":["up_movie_prod", "movie_id", "movie_prod"], "user_list_db":["up_user", "movie_id", "user_list"]}

In [88]:
for key, value in dict_file.items():
    if os.path.isfile(f"update_db/{value[0]}.csv"):
        database = pd.read_csv(f"db_backup/{key}.csv",encoding="ISO-8859-1", sep=";").set_index(value[1])
        up = pd.read_csv(f"update_db/{value[0]}.csv",encoding="ISO-8859-1", sep=";").set_index(value[1])
        up.to_sql(value[2], engine, if_exists="append")
        pd.concat([database, up]).to_csv(f"db_backup/{key}.csv",encoding="ISO-8859-1", sep=";")